In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from src.features import build_feature_creator, build_column_transformer

In [2]:
print("hello")

hello


In [3]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.features import build_preprocessor

train = pd.read_csv("../donnees/processed/train.csv")
X_train = train.drop(columns="Class")
y_train = train["Class"]

print(X_train.shape, y_train.sum())

(227845, 30) 394


In [4]:
from imblearn.pipeline import Pipeline as ImbPipeline


In [5]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Poids de la classe positive pour XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight :", round(scale_pos_weight, 1))

scale_pos_weight : 577.3


In [6]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [7]:
from src.features import build_feature_creator, build_column_transformer

pipelines = {
    # --- Random Forest ---
    "RF_class_weight": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("model", RandomForestClassifier(
            class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1
        )),
    ]),
    "RF_smote": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
    "RF_undersample": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("under", RandomUnderSampler(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),

    # --- XGBoost ---
    "XGB_scale_pos_weight": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("model", XGBClassifier(
            scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
            random_state=42, n_jobs=-1
        )),
    ]),
    "XGB_smote": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("smote", SMOTE(random_state=42)),
        ("model", XGBClassifier(eval_metric="aucpr", random_state=42, n_jobs=-1)),
    ]),
    "XGB_undersample": ImbPipeline([
        ("features", build_feature_creator()),
        ("prep", build_column_transformer()),
        ("under", RandomUnderSampler(random_state=42)),
        ("model", XGBClassifier(eval_metric="aucpr", random_state=42, n_jobs=-1)),
    ]),
}

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import time

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, pipe in pipelines.items():
    start = time.time()
    scores = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1
    )
    duration = time.time() - start
    results[name] = scores
    print(f"{name:22s} PR-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})  [{duration:.0f}s]")

RF_class_weight        PR-AUC = 0.8443 (+/- 0.0233)  [134s]
RF_smote               PR-AUC = 0.8467 (+/- 0.0282)  [675s]
RF_undersample         PR-AUC = 0.7391 (+/- 0.0304)  [2s]
XGB_scale_pos_weight   PR-AUC = 0.8463 (+/- 0.0197)  [19s]
XGB_smote              PR-AUC = 0.8450 (+/- 0.0227)  [30s]
XGB_undersample        PR-AUC = 0.7416 (+/- 0.0494)  [7s]


In [9]:
import sys
sys.path.append("..")

from src.features import build_feature_creator, build_column_transformer


In [10]:
from src.features import build_feature_creator, build_column_transformer

print(type(build_feature_creator()))
print(type(build_column_transformer()))

<class 'sklearn.preprocessing._function_transformer.FunctionTransformer'>
<class 'sklearn.compose._column_transformer.ColumnTransformer'>
